# Classificação de Auxílios FAPESP com Gemini 2.5 Flash Lite

Este notebook gera uma versão simplificada do conjunto de **Auxílios em Andamento** da FAPESP e classifica automaticamente quais pesquisadores são potenciais clientes com base no **Resumo em Português** de cada projeto.

> ⚠️ **Pré-requisitos**
> - Possuir um arquivo CSV exportado do portal da FAPESP contendo pelo menos as colunas `Beneficiário` (ou `Pesquisador Responsável`) e `Resumo (Português)`.
> - Ter a biblioteca [`google-generativeai`](https://pypi.org/project/google-generativeai/) instalada.
> - Definir a variável de ambiente `GOOGLE_API_KEY` com uma chave válida do Gemini ou informar a chave manualmente na célula de configuração.


## 1. Configuração
Preencha o caminho do CSV de entrada e, se necessário, ajuste os hiperparâmetros. Caso a variável de ambiente `GOOGLE_API_KEY` não esteja definida, informe a chave manualmente na variável `MANUAL_API_KEY`.


In [ ]:
import os
from pathlib import Path

import pandas as pd

# --- Configuração da chave da API -----------------------------------------------------------
MANUAL_API_KEY = ""  # opcional: cole aqui a chave apenas durante a execução local
api_key = os.getenv("GOOGLE_API_KEY") or MANUAL_API_KEY
if not api_key:
    raise RuntimeError("Defina GOOGLE_API_KEY no ambiente ou preencha MANUAL_API_KEY.")
os.environ["GOOGLE_API_KEY"] = api_key

# --- Caminhos e parâmetros principais -------------------------------------------------------
MODEL_NAME = "gemini-2.5-flash-lite"
INPUT_CSV_PATH = Path("/content/auxilios_em_andamento.csv")
OUTPUT_NAME = "classificacao_auxilios"
OUTPUT_DIR = Path("./data/output/gemini_classification")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV_PATH = OUTPUT_DIR / f"{OUTPUT_NAME}.csv"
CLIENTS_CSV_PATH = OUTPUT_DIR / f"{OUTPUT_NAME}_clientes.csv"
NON_CLIENTS_CSV_PATH = OUTPUT_DIR / f"{OUTPUT_NAME}_nao_clientes.csv"

BATCH_SIZE = 15
MAX_RETRIES = 3
SLEEP_BETWEEN_RETRIES = 5  # segundos
TEMPERATURE = 0.0

print(f"Usando modelo: {MODEL_NAME}")
print(f"Arquivo de entrada: {INPUT_CSV_PATH}")
print(f"Resultados serão salvos em: {OUTPUT_CSV_PATH}")


## 2. Carregamento e preparação do DataFrame simplificado (DF2)
A célula abaixo lê o CSV original, seleciona os campos relevantes e produz o DataFrame `df2` contendo apenas o nome do pesquisador e o resumo em português. Os nomes das colunas podem variar entre exportações; o código tenta usar automaticamente a opção disponível.


In [ ]:
import unicodedata

def normalizar_coluna(nome: str) -> str:
    nome = unicodedata.normalize("NFKD", nome).encode("ascii", "ignore").decode("ascii")
    return nome.strip().lower()

RAW_NAME_COLUMNS = {
    "beneficiario",
    "pesquisador responsavel",
}
RESUMO_COLUMNS = {
    "resumo (portugues)",
    "resumo em portugues",
}

# Tenta detectar o separador automaticamente (CSV da FAPESP costuma usar ponto e vírgula)
try:
    df_raw = pd.read_csv(INPUT_CSV_PATH)
except Exception:
    df_raw = pd.read_csv(INPUT_CSV_PATH, sep=";", encoding="utf-8")

colunas_norm = {normalizar_coluna(col): col for col in df_raw.columns}

coluna_nome = next((colunas_norm[c] for c in RAW_NAME_COLUMNS if c in colunas_norm), None)
coluna_resumo = next((colunas_norm[c] for c in RESUMO_COLUMNS if c in colunas_norm), None)

if not coluna_nome or not coluna_resumo:
    raise KeyError(
        "Não foi possível encontrar as colunas necessárias."
        f" Encontradas: {list(df_raw.columns)}"
    )

print(f"Coluna de nome detectada: {coluna_nome}")
print(f"Coluna de resumo detectada: {coluna_resumo}")

df2 = (
    df_raw[[coluna_nome, coluna_resumo]]
    .rename(columns={coluna_nome: "nome_pesquisador", coluna_resumo: "resumo_portugues"})
    .dropna(subset=["resumo_portugues"])
)

# Remove espaços extras e entradas duplicadas mantendo o resumo mais recente

df2["nome_pesquisador"] = df2["nome_pesquisador"].astype(str).str.strip()
df2["resumo_portugues"] = df2["resumo_portugues"].astype(str).str.strip()

df2 = df2.drop_duplicates(subset=["nome_pesquisador", "resumo_portugues"])

print(f"Total de projetos carregados: {len(df_raw):,}")
print(f"Projetos após limpeza: {len(df2):,}")
df2.head()


## 3. Configuração do modelo Gemini e funções auxiliares
Esta célula inicializa o cliente do Gemini e define as funções auxiliares utilizadas na classificação:

- `build_prompt(batch)` cria o prompt para cada lote.
- `call_gemini(prompt)` executa a chamada ao modelo com tratamento de *retry*.
- `parse_response(text)` converte a resposta do LLM em um dicionário estruturado.
- `classificar_batch(batch)` coordena o fluxo completo para um lote.

A heurística considera um pesquisador **CLIENTE** se o resumo estiver alinhado com as categorias positivas fornecidas. Qualquer outro caso recebe a etiqueta **NAO_CLIENTE**. Palavras-chave detectadas automaticamente podem ser usadas como *fallback* caso o modelo retorne uma resposta vazia ou mal formatada.


In [ ]:
import json
import time
from typing import Iterable, List, Dict

import google.generativeai as genai

POSITIVE_CATEGORIES = [
    "sintese de gene",
    "producao de proteina",
    "antigeno",
    "elisa",
    "biologia molecular",
    "biologia celular",
    "bioquimica",
    "anticorpo",
    "proteomica",
    "expressao proteica",
    "engenharia genetica",
    "diagnostico molecular",
]

POSITIVE_KEYWORDS = [c.lower() for c in POSITIVE_CATEGORIES]

SYSTEM_RULES = """
Você é um assistente técnico que classifica resumos de projetos financiados pela FAPESP.
Retorne apenas JSON válido seguindo o formato solicitado, sem texto adicional.
Classifique como "CLIENTE" quando o projeto tratar de proteínas, antígenos, ensaios imunoquímicos, biologia molecular/celular ou tópicos correlatos das categorias positivas fornecidas.
Caso contrário, classifique como "NAO_CLIENTE".
Sempre inclua uma justificativa curta mencionando as evidências do resumo.
""".strip()

genai.configure(api_key=api_key)
model = genai.GenerativeModel(
    model_name=MODEL_NAME,
    system_instruction=SYSTEM_RULES,
    generation_config={
        "temperature": TEMPERATURE,
        "top_p": 1,
        "top_k": 32,
        "max_output_tokens": 2048,
        "response_mime_type": "application/json",
    },
)

def build_prompt(batch: Iterable[Dict[str, str]]) -> str:
    payload = [
        {
            "nome_pesquisador": item["nome_pesquisador"],
            "resumo_portugues": item["resumo_portugues"],
            "categorias_positivas": POSITIVE_CATEGORIES,
            "formato_resposta": {
                "nome_pesquisador": "string",
                "classificacao": "CLIENTE|NAO_CLIENTE",
                "categorias_identificadas": ["string"],
                "justificativa": "string",
            },
        }
        for item in batch
    ]
    instructions = {
        "tarefa": "Classifique cada pesquisador como CLIENTE ou NAO_CLIENTE.",
        "criterio": "CLIENTE se o resumo se relacionar às categorias positivas.",
        "saida": "Retorne uma lista JSON com um objeto por pesquisador na mesma ordem recebida.",
        "dados": payload,
    }
    return json.dumps(instructions, ensure_ascii=False, indent=2)


def call_gemini(prompt: str) -> str:
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = model.generate_content(prompt, request_options={"timeout": 600})
            if not response or not response.text:
                raise ValueError("Resposta vazia do modelo.")
            return response.text
        except Exception as exc:  # pylint: disable=broad-except
            last_error = exc
            print(f"Tentativa {attempt}/{MAX_RETRIES} falhou: {exc}")
            if attempt < MAX_RETRIES:
                time.sleep(SLEEP_BETWEEN_RETRIES)
    raise RuntimeError(f"Falha após {MAX_RETRIES} tentativas") from last_error


def parse_response(text: str) -> List[Dict[str, str]]:
    text = text.strip()
    if text.startswith("{") and text.endswith("}"):
        data = json.loads(text)
        if isinstance(data, dict) and "resultados" in data:
            data = data["resultados"]
    else:
        start = text.find("[")
        end = text.rfind("]")
        if start != -1 and end != -1:
            text = text[start : end + 1]
        data = json.loads(text)
    if not isinstance(data, list):
        raise ValueError("Resposta não contém uma lista JSON.")
    parsed = []
    for item in data:
        parsed.append(
            {
                "nome_pesquisador": item.get("nome_pesquisador", "").strip(),
                "classificacao": item.get("classificacao", "").strip().upper(),
                "categorias_identificadas": item.get("categorias_identificadas", []),
                "justificativa": item.get("justificativa", "").strip(),
            }
        )
    return parsed


def fallback_keyword_classification(resumo: str) -> Dict[str, str]:
    resumo_lower = resumo.lower()
    categorias_detectadas = [
        palavra for palavra in POSITIVE_KEYWORDS if palavra in resumo_lower
    ]
    if categorias_detectadas:
        classificacao = "CLIENTE"
        justificativa = (
            "Palavras-chave positivas detectadas: "
            + ", ".join(sorted(categorias_detectadas))
        )
    else:
        classificacao = "NAO_CLIENTE"
        justificativa = "Nenhuma palavra-chave positiva encontrada no resumo."
    return {
        "classificacao": classificacao,
        "categorias_identificadas": categorias_detectadas,
        "justificativa": justificativa,
    }


def classificar_batch(batch: List[Dict[str, str]]) -> List[Dict[str, str]]:
    prompt = build_prompt(batch)
    try:
        resposta = call_gemini(prompt)
        resultados = parse_response(resposta)
    except Exception as exc:  # pylint: disable=broad-except
        print(f"Falha ao processar lote com LLM, aplicando fallback: {exc}")
        resultados = []

    if not resultados or len(resultados) != len(batch):
        # Fallback parcial com base em palavras-chave
        resultados = []
        for item in batch:
            fallback = fallback_keyword_classification(item["resumo_portugues"])
            resultados.append({
                "nome_pesquisador": item["nome_pesquisador"],
                **fallback,
            })
    else:
        # Ajuste: completa campos faltantes
        completos = []
        for item, resultado in zip(batch, resultados):
            fallback = fallback_keyword_classification(item["resumo_portugues"])
            classificacao = resultado.get("classificacao") or fallback["classificacao"]
            if classificacao not in {"CLIENTE", "NAO_CLIENTE"}:
                classificacao = fallback["classificacao"]
            categorias = resultado.get("categorias_identificadas") or fallback["categorias_identificadas"]
            justificativa = resultado.get("justificativa") or fallback["justificativa"]
            completos.append({
                "nome_pesquisador": item["nome_pesquisador"],
                "classificacao": classificacao,
                "categorias_identificadas": categorias,
                "justificativa": justificativa,
            })
        resultados = completos
    return resultados


## 4. Execução da classificação
Esta etapa percorre o DataFrame simplificado em lotes (`BATCH_SIZE`), consulta o modelo e agrega os resultados finais. Ao término, o notebook grava três arquivos CSV:

1. `classificacao_auxilios.csv`: conjunto completo com classificação, categorias e justificativa.
2. `classificacao_auxilios_clientes.csv`: apenas pesquisadores classificados como **CLIENTE**.
3. `classificacao_auxilios_nao_clientes.csv`: pesquisadores classificados como **NAO_CLIENTE** (útil para revisão posterior).

O DataFrame final também é exibido para inspeção rápida.


In [ ]:
resultados = []

lote_atual = []
for _, row in df2.iterrows():
    registro = {
        "nome_pesquisador": row["nome_pesquisador"],
        "resumo_portugues": row["resumo_portugues"],
    }
    lote_atual.append(registro)
    if len(lote_atual) == BATCH_SIZE:
        resultados.extend(classificar_batch(lote_atual))
        lote_atual = []

if lote_atual:
    resultados.extend(classificar_batch(lote_atual))

resultado_df = pd.DataFrame(resultados)
resultado_df = resultado_df.merge(df2, on="nome_pesquisador", how="left")

resultado_df.to_csv(OUTPUT_CSV_PATH, index=False)
resultado_df.query("classificacao == 'CLIENTE'").to_csv(CLIENTS_CSV_PATH, index=False)
resultado_df.query("classificacao == 'NAO_CLIENTE'").to_csv(NON_CLIENTS_CSV_PATH, index=False)

print("Resumo das classificações:")
print(resultado_df["classificacao"].value_counts())
print("
Arquivos gerados:")
print(f"- Completo: {OUTPUT_CSV_PATH}")
print(f"- Clientes: {CLIENTS_CSV_PATH}")
print(f"- Não clientes: {NON_CLIENTS_CSV_PATH}")

resultado_df.head()


## 5. Próximos passos sugeridos
- Revisar manualmente uma amostra dos casos classificados como **NAO_CLIENTE** para garantir que nenhum cliente relevante foi descartado.
- Integrar a etapa de busca de e-mails utilizando o recurso *browser-use* do Gemini, empregando o CSV de clientes como entrada.
- Ajustar o prompt adicionando novas categorias positivas ou negativas conforme feedback da equipe de negócios.
